# Mini-ColBERTv2 on Multiple BEIR Datasets (Colab)

Runs the base -> cad -> evaluation pipeline on several BEIR datasets to check generality:

| Dataset | Docs | Queries | Domain |
|---|---|---|---|
| NFCorpus | 3.6k | 323 | Biomedical |
| SCIDOCS | 25k | 1000 | Scientific |
| FiQA-2018 | 57k | 648 | Finance |
| ArguAna | 8.7k | 1406 | Argumentation |

Each dataset runs independently; heavy outputs stay local in `outputs/<dataset>/`.

**Google Drive (recommended):** the notebook mounts Drive and syncs only the
SMALL result files (`rankings_*.json`, `qrels_test.json`, `base_candidates.pt`,
`official_baseline.json`) into `MyDrive/colbert-run/<dataset>/`. This avoids
re-running encode/index (~20-25 min for 4 datasets) on every new session while
still letting you print the table.

The `Official ColBERTv2` row comes from
`MyDrive/colbert-run/<dataset>/official_baseline.json`, produced by
[`Official_ColBERTv2_MultiBEIR.ipynb`](Official_ColBERTv2_MultiBEIR.ipynb).

(TREC-COVID has 171k docs and is heavy, so it is excluded by default.)

In [ ]:
# 1. Clone repo
!git clone https://github.com/HoangNguyen689/colbert-hoangnd.git || true
%cd colbert-hoangnd
!git pull

In [ ]:
# 2. Install dependencies (pin transformers<5 to stay compatible with colbert-ai)
!pip install -q "transformers==4.57.6" "colbert-ai==0.2.22" scikit-learn datasets beir tqdm numpy ninja

In [ ]:
# 3. Ensure ninja is on PATH for colbert-ai C++ extension compilation
import os, sys
os.environ['PATH'] += ':' + os.path.join(sys.prefix, 'bin')
!ninja --version

In [ ]:
# 4. Detect device
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# 5. Mount Google Drive to persist SMALL result files across sessions
USE_DRIVE = True
DRIVE_DIR = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/colbert-run'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print('Drive dir:', DRIVE_DIR)

In [ ]:
# 6. Run full pipeline per dataset (local, fast) and sync SMALL files to Drive
import shutil
from pathlib import Path

DATASETS = ['nfcorpus', 'scidocs', 'fiqa', 'arguana']
SMALL_FILES = ['rankings_original.json', 'rankings_idf.json',
               'rankings_adaptive.json', 'rankings_cad.json',
               'qrels_test.json', 'base_candidates.pt']

for ds in DATASETS:
    out = Path(f'outputs/{ds}')
    print(f'\n==================== {ds} ====================')
    !python run_base.py --dataset {ds} --output_dir {out} --K 8192 --n_samples 200000 --ncells 4 --candidate_topn 200 --topk 100 --device {DEVICE} --doc_batch_size 64 --query_batch_size 64
    !python run_cad.py --dataset {ds} --output_dir {out} --topk 100

    # Pull the official baseline json produced by the Official notebook (small, on Drive)
    if DRIVE_DIR:
        src = Path(DRIVE_DIR) / ds / 'official_baseline.json'
        if src.exists():
            shutil.copy(src, out / 'official_baseline.json')
            print('Loaded official baseline from Drive')

    !python run_evaluate.py --dataset {ds} --output_dir {out} --qrels {out}/qrels_test.json

    # Push small result files back to Drive
    if DRIVE_DIR:
        dst = Path(DRIVE_DIR) / ds
        dst.mkdir(parents=True, exist_ok=True)
        for name in SMALL_FILES:
            p = out / name
            if p.exists():
                shutil.copy(p, dst / name)
        print('Synced small files ->', dst)

In [ ]:
# 7. Aggregate results across datasets into one comparison table
#    Reads local outputs/ if present, otherwise the Drive copy.
import json
from pathlib import Path
from evaluation.metrics import evaluate, load_json

VARIANTS = ['official', 'original', 'idf', 'adaptive', 'cad']
LABELS = {'official': 'Official', 'original': 'Mini Base', 'idf': 'IDF',
          'adaptive': 'Adaptive', 'cad': 'CAD'}

def resolve(ds):
    local = Path(f'outputs/{ds}')
    if (local / 'rankings_original.json').exists():
        return local
    if DRIVE_DIR:
        d = Path(DRIVE_DIR) / ds
        if (d / 'rankings_original.json').exists():
            return d
    return local

rows = {}
for ds in DATASETS:
    d = resolve(ds)
    qrels_path = d / 'qrels_test.json'
    if not qrels_path.exists():
        continue
    qrels = load_json(qrels_path)
    off = d / 'official_baseline.json'
    if off.exists():
        rows[(ds, 'official')] = json.loads(off.read_text())
    for name in ['original', 'idf', 'adaptive', 'cad']:
        path = d / f'rankings_{name}.json'
        if path.exists():
            rows[(ds, name)] = evaluate(load_json(path), qrels)

print(f"{'Dataset':<14} {'Method':<10} {'nDCG@10':>8} {'MRR@10':>8} {'Recall@100':>11}")
print('-' * 55)
for ds in DATASETS:
    for name in VARIANTS:
        m = rows.get((ds, name))
        if m:
            print(f"{ds:<14} {LABELS[name]:<10} {m['nDCG@10']:>8.4f} {m['MRR@10']:>8.4f} {m['Recall@100']:>11.4f}")
    print('-' * 55)

In [ ]:
# 8. (Optional) Grid-search beta for CAD on each dataset
# for ds in DATASETS:
#     out = f'outputs/{ds}'
#     !python -m cad.tune_cad --candidates {out}/base_candidates.pt --query_emb {out}/query_embeddings.pt --idf {out}/idf.pt --qrels {out}/qrels_test.json --output_dir {out} --topk 100
pass